### Воробьёва Юлия, 232

#### Подготовка

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("human.csv", sep="\t")
df["label"] = "human"
df["text_id"] = df["Filename"].apply(
    lambda x: int(re.search(r"_(\d+)\.conllu", x).group(1))
)
human = df.copy()

print(f"Human dataset shape: {human.shape}")

df = pd.read_csv("chatgpt.csv", sep="\t")
df["label"] = "chatgpt"
df["text_id"] = df["Filename"].apply(
    lambda x: int(re.search(r"_(\d+)\.conllu", x).group(1))
)
chatgpt = df.copy()

print(f"ChatGPT dataset shape: {chatgpt.shape}")

df = pd.read_csv("mixed.csv", sep="\t")
df["label"] = "mixed"
df["text_id"] = df["Filename"].apply(
    lambda x: int(re.search(r"_(\d+)\.conllu", x).group(1))
)
mixed = df.copy()

print(f"Mixed dataset shape: {mixed.shape}")

Human dataset shape: (3607, 130)
ChatGPT dataset shape: (3607, 131)
Mixed dataset shape: (3607, 130)


In [3]:
feature_cols = sorted(set(human.columns) | set(chatgpt.columns) | set(mixed.columns)
                           - {"Filename", "label", "text_id"})

def align(df):
    for c in feature_cols:
        if c not in df.columns:
            df[c] = 0.0
    return df[["Filename", "label", "text_id"] + feature_cols]

human = align(human)
chatgpt = align(chatgpt)
mixed = align(mixed)

data = pd.concat([human, chatgpt, mixed], ignore_index=True)
data = data.loc[:, ~data.columns.duplicated()]
numeric_cols = data.select_dtypes(include="number").columns.tolist()

print(f"Итоговая таблица: ", data.shape)
print("Числовых признаков: ", len(numeric_cols))
data.to_csv("all_data.csv", index=False)

data.head()

Итоговая таблица:  (10821, 132)
Числовых признаков:  130


,Filename,label,text_id,aux_form_dist_Fin,aux_form_dist_Ger,aux_form_dist_Inf,aux_form_dist_Part,aux_mood_dist_Imp,aux_mood_dist_Ind,aux_num_pers_dist_+,...,verbs_form_dist_Ger,verbs_form_dist_Inf,verbs_form_dist_Part,verbs_mood_dist_Imp,verbs_mood_dist_Ind,verbs_num_pers_dist_+,verbs_num_pers_dist_Sing+,verbs_num_pers_dist_Sing+3,verbs_tense_dist_Past,verbs_tense_dist_Pres
0,human/human_1852.conllu,human,1852,80.000000,0.0,0.000000,20.000000,0.0,100.0,50.000000,...,29.166667,20.833333,16.666667,62.500000,37.500000,62.5,0.0,37.5,57.142857,42.857143
1,human/human_3417.conllu,human,3417,81.818182,0.0,0.000000,18.181818,0.0,100.0,88.888889,...,30.434783,26.086957,17.391304,16.666667,83.333333,100.0,0.0,0.0,77.777778,22.222222
2,human/human_2117.conllu,human,2117,70.833333,0.0,20.833333,8.333333,0.0,100.0,58.823529,...,17.647059,38.235294,20.588235,0.000000,100.000000,62.5,0.0,37.5,60.000000,40.000000
3,human/human_2345.conllu,human,2345,100.000000,0.0,0.000000,0.000000,0.0,100.0,20.000000,...,25.000000,0.000000,41.666667,0.000000,100.000000,75.0,0.0,25.0,88.888889,11.111111
4,human/human_1342.conllu,human,1342,66.666667,0.0,33.333333,0.000000,0.0,100.0,50.000000,...,40.000000,20.000000,20.000000,0.000000,100.000000,0.0,0.0,100.0,50.000000,50.000000


#### Корреляционный анализ

Сначала посчитаем матрицу Пирсона между всеми признаками, выведем пары, которые коррелируют почти на 1

In [4]:
corr_matrix = data[numeric_cols].corr()

high_corr = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)))
high_corr = high_corr.stack().reset_index()
high_corr.columns = ["feature_1", "feature_2", "correlation"]
high_corr = high_corr[high_corr["correlation"].abs() > 0.9].sort_values(by="correlation", key=abs, ascending=False)
print(f"Признаки с высокой корреляцией: {len(high_corr)}")
high_corr.head(15)

Признаки с высокой корреляцией: 14


,feature_1,feature_2,correlation
7016,principal_proposition_dist,subordinate_proposition_dist,-1.000000
8384,verbs_tense_dist_Past,verbs_tense_dist_Pres,-1.000000
7059,subj_post,subj_pre,-1.000000
7606,ttr_form_chunks_200,ttr_lemma_chunks_200,0.999465
6082,dep_dist_punct,upos_dist_PUNCT,0.998626
4258,dep_dist_det,upos_dist_DET,0.995009
7566,ttr_form_chunks_100,ttr_lemma_chunks_100,0.992659
3501,dep_dist_cc,upos_dist_CCONJ,0.982127
7395,subordinate_post,subordinate_pre,-0.980781
3399,dep_dist_case,upos_dist_ADP,0.951035


Теперь посмотрим на информативные признаки

In [5]:
from scipy.stats import f_oneway

results = []

for feat in numeric_cols:
    human = data[data["label"] == "human"][feat]
    chatgpt = data[data["label"] == "chatgpt"][feat]
    mixed = data[data["label"] == "mixed"][feat]

    F, p = f_oneway(human, chatgpt, mixed)

    results.append([feat, F, p])

results = pd.DataFrame(
    results,
    columns=["feature", "F", "p_value"]
)

results = results.sort_values("p_value")

results.head(20)

,feature,F,p_value
20,char_per_tok,1141.840067,0.000000e+00
96,upos_dist_AUX,752.216332,1.330092e-306
120,verbs_form_dist_Ger,581.547604,1.298154e-240
36,dep_dist_cop,570.466156,2.902456e-236
65,lexical_density,462.521897,1.808415e-193
62,dep_dist_root,372.299339,4.305846e-157
68,n_sentences,314.158743,2.386455e-133
23,dep_dist_advcl,290.373485,1.448350e-123
56,dep_dist_obj,205.374169,2.875647e-88
28,dep_dist_aux:pass,204.047713,1.032288e-87


Для каждого признака я провела однофакторный дисперсионный анализ ANOVA и сравнила p-value между тремя классами текстов. Признаки с низким p-value считаются статистически значимо различающимися между классами